# ticket-triage: fine-tune xlm-roberta-base (three heads) on a Kaggle T4

Runs headless with **Save Version → Save & Run All**. Settings: **Accelerator GPU T4 x2** (one GPU is used), **Internet on**,
and a **Secret `HF_TOKEN`** with write access to `DavutKursun/ticket-triage-xlmr` (Add-ons → Secrets). Without the secret
the push is skipped and the files stay in the Output tab.

Steps: GPU check → shallow clone of the public repo → `pip install -r requirements-kaggle.txt` (never torch) →
`scripts/train.py --model-name xlm-roberta-base` → `scripts/export_onnx.py` → `scripts/model_card.py` → push to the Hub.
The same cells run locally on CPU with a tiny model via `scripts/notebook_smoke.py`, so this is never the first run.

In [ ]:
# Settings
import json, os, shutil, subprocess, sys, time
from pathlib import Path

SMOKE = os.environ.get("TT_SMOKE") == "1"  # local CPU dry run (scripts/notebook_smoke.py); never set on Kaggle
REPO_URL = "https://github.com/DavutKursun/ticket-triage.git"
HUB_REPO = "DavutKursun/ticket-triage-xlmr"
WORK = Path(os.environ.get("TT_WORK", "/kaggle/working"))  # Kaggle saves this folder as the notebook output
REPO = Path(os.environ["TT_REPO"]) if SMOKE else Path("/tmp/ticket-triage")  # code checkout, not part of the output
OUT = WORK / "model"
PY = sys.executable
TRAIN_ARGS = ["--tiny", "--limit", "64", "--epochs", "1"] if SMOKE else ["--model-name", "xlm-roberta-base"]
EXPORT_ARGS = ["--limit", "200", "--latency-n", "40"] if SMOKE else []
UPLOAD = ["model.safetensors", "model.pt", "encoder/*", "tokenizer/*", "labels.json", "ft_config.json", "dev_metrics.json",
          "train_meta.json", "export_report.json", "onnx/*", "README.md"]  # never fp32/: the fp32 ONNX stays local
os.environ.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
T0 = time.time()


def run(cmd, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)


print("smoke run" if SMOKE else "Kaggle run", "| output folder:", OUT)

In [ ]:
# 1. GPU and versions
import platform

import torch

if shutil.which("nvidia-smi"):
    run(["nvidia-smi"])
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPUs", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not SMOKE and not torch.cuda.is_available():
    raise RuntimeError("No GPU: set Accelerator to GPU T4 x2 in the notebook settings and run again.")

In [ ]:
# 2. Code: shallow clone of the public repo (the commit also lands in train_meta.json)
if not SMOKE:
    shutil.rmtree(REPO, ignore_errors=True)
    run(["git", "clone", "--depth", "1", REPO_URL, REPO])
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip()
run(["git", "log", "-1", "--format=%H %cd %s", "--date=iso"], cwd=REPO)

In [ ]:
# 3. Packages Kaggle lacks or that must match the tested versions (never torch)
if SMOKE:
    print("smoke run: the local .venv-train already has requirements-train.txt")
else:
    run([PY, "-m", "pip", "install", "-q", "-r", REPO / "requirements-kaggle.txt"])
run([PY, "-c", "import transformers, tokenizers, onnxruntime, huggingface_hub; print('transformers', transformers.__version__, "
     "'| tokenizers', tokenizers.__version__, '| onnxruntime', onnxruntime.__version__, '| huggingface_hub', huggingface_hub.__version__)"])

In [ ]:
# 4. Fine-tune: dev scored after every epoch, best epoch kept (about 15-25 min on one T4)
shutil.rmtree(OUT, ignore_errors=True)
run([PY, "scripts/train.py", *TRAIN_ARGS, "--out", OUT], cwd=REPO)
dev = json.loads((OUT / "dev_metrics.json").read_text())
print("best epoch", dev["best_epoch"], {t: dev["best"][t]["macro_f1"] for t in ("category", "urgency", "sentiment")})

In [ ]:
# 5. ONNX export: fp32 in fp32/ (stays here), int8 in onnx/ (served), parity report
run([PY, "scripts/export_onnx.py", OUT, *EXPORT_ARGS], cwd=REPO)
rep = json.loads((OUT / "export_report.json").read_text())
print("exporter", rep["exporter"], "| int8", rep["int8"]["kept"], "| agreement", rep["int8"]["argmax_agreement"],
      "| tokenizer mismatches", rep["tokenizer_parity"]["mismatches"])

In [ ]:
# 6a. Model card (README.md for the Hub)
run([PY, "scripts/model_card.py", OUT, "--repo-id", HUB_REPO], cwd=REPO)

In [ ]:
# 6b. Push to the Hub with the Kaggle Secret HF_TOKEN (read, never printed). Without it: download from the Output tab.
token = None
if not SMOKE:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN") or None
    except Exception as e:  # secret missing or not attached to this notebook
        print("HF_TOKEN secret not available:", type(e).__name__)
if token:
    from huggingface_hub import HfApi

    info = HfApi(token=token).upload_folder(folder_path=str(OUT), repo_id=HUB_REPO, repo_type="model", allow_patterns=UPLOAD,
                                            commit_message=f"Train run from ticket-triage@{COMMIT[:7]}")
    token = None
    print("pushed to", f"https://huggingface.co/{HUB_REPO}", "|", getattr(info, "commit_url", ""))
else:
    files = sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file() and not p.relative_to(OUT).as_posix().startswith("fp32/"))
    print("Push skipped. Download these from the Output tab (folder model/), or add the HF_TOKEN secret and run again:")
    for f in files:
        print("  model/" + f)
print(f"total {(time.time() - T0) / 60:.1f} min")